# Домашнее задание: Вычисление MFCC и работа с фазой аудио
В этом ноутбуке мы:
1. Вручную вычислим MFCC-признаки на PyTorch (STFT → Mel-фильтры → Логарифм → ДКП).
2. Поэкспериментируем с фазой спектра: замена на случайную, подстановка фазы из другого сигнала и восстановление фазы алгоритмом Griffin-Lim.

In [ ]:
import torch
import torchaudio

import matplotlib.pyplot as plt
from IPython import display

Для воспроизведения в Jupyter

In [ ]:
def visualize_audio(wav: torch.Tensor, sr: int = 22050):
    # Average all channels
    if wav.dim() == 2:
        # Any to mono audio convertion
        wav = wav.mean(dim=0)

    plt.figure(figsize=(20, 5))
    plt.plot(wav, alpha=.7, c='green')
    plt.grid()
    plt.xlabel('Time', size=20)
    plt.ylabel('Amplitude', size=20)
    plt.show()

    display.display(display.Audio(wav, rate=sr))

In [ ]:
# Параметры
SR = 24000
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
WINDOW = torch.hann_window(WIN_LENGTH)

Загрузим основную аудиозапи

In [ ]:
wav, sr = torchaudio.load('speaker_1.wav')

if sr != SR:
    wav = torchaudio.transforms.Resample(sr, SR)(wav)

In [ ]:
visualize_audio(wav, sr)

## 1. Расчёт MFCC на PyTorch
Алгоритм вычисления:
1. **STFT** → комплексный спектр
2. **Power Spectrogram** $S(t, f) = |X(t, f)|^2$
3. **Mel Filterbank** → проекция на шкалу Мэла (треугольные фильтры)
4. **Logarithm** → $\log_{10}(\text{MelSpec} + \epsilon)$
5. **DCT-II** → декорреляция коэффициентов (получаем MFCC)

Вычислите комплексную спектр STFT с заданными начальными параметрами (N_FFT, HOP_SIZE, и т.д.)

In [ ]:
spectrum = None

Вычислите амплитуду спектра

In [ ]:
magnitude = None

Визуализируем спектрограмму

In [ ]:
plt.figure(figsize=(10, 3))
plt.imshow(magnitude.log().squeeze(), aspect='auto', origin='lower')
plt.title("Амплитуда аудиофайла")
plt.colorbar()
plt.xlabel("Кадр")
plt.ylabel("Частота")
plt.show()

Получения фильт-банков для вычисление мел-спектрограммы

In [ ]:
mel_scaler = None

Вычисление мел-спектрограммы

In [ ]:
mel_spectrogram = mel_scaler(magnitude)

In [ ]:
plt.figure(figsize=(20, 5))
plt.imshow(mel_spectrogram.squeeze().log(), origin='lower')
plt.xlabel('Time', size=20)
plt.ylabel('Mels', size=20)
plt.show()

### Математическое описание DCT-матрицы для MFCC

Для декорреляции логарифмированных энергий Mel-фильтров используется **Дискретное Косинусное Преобразование II типа (DCT-II)**.

Пусть $N = \text{n\_filters}$ — количество Mel-полос, а $K = \text{n\_mfcc}$ — целевое число коэффициентов. Матрица преобразования $C \in \mathbb{R}^{K \times N}$ строится по формуле:
$$ C_{k, n} = c_k \cdot \cos\left( \frac{\pi \cdot k \cdot (n + 0.5)}{N} \right), \quad \begin{cases} k = 0, \dots, K-1 \\ n = 0, \dots, N-1 \end{cases} $$

**Нормализация** (обеспечивает ортонормированность базиса):
$$ c_k = \begin{cases} \sqrt{\dfrac{1}{N}}, & k = 0 \\ \sqrt{\dfrac{2}{N}}, & k > 0 \end{cases} $$

**Применение к спектрограмме:**
Если $E \in \mathbb{R}^{N \times T}$ — логарифмированная Mel-спектрограмма, то MFCC-коэффициенты вычисляются матричным умножением:
$$ \text{MFCC} = C \cdot \log(E + \epsilon) $$
Результат $\text{MFCC} \in \mathbb{R}^{K \times T}$ содержит декоррелированные признаки. Первый коэффициент ($k=0$) отражает общую энергию кадра, а остальные ($k=1\dots K-1$) описывают форму спектра и обычно используются в ML-моделях.

In [ ]:
import math

def get_dct_matrix(n_filters, n_mfcc):
    n = torch.arange(n_filters).float()
    # Используем unsqueeze, чтобы подготовить матрицу для broadcasting
    k = torch.arange(n_mfcc).float().unsqueeze(1)

    weights = None

    # Нормализация для ортонормированности
    weights *= math.sqrt(2.0 / n_filters)
    weights[0] /= math.sqrt(2.0)  # Корректировка k=0

    return weights

In [ ]:
# 1. Берем логарифм (с защитой от log(0))
log_mel_spec = torch.log(mel_spectrogram + 1e-6)

# 2. Генерируем матрицу DCT весов
n_mels = mel_spectrogram.size(-2)
dct_mat = get_dct_matrix(n_mels, 13)

# 3. Применяем DCT через матричное умножение
# [n_mfcc, n_mels] x [..., n_mels, time] -> [..., n_mfcc, time]
mfcc = None

In [ ]:
plt.figure(figsize=(12, 4))
plt.imshow(
    mfcc.squeeze(),
    aspect='auto',
    origin='lower',
    #cmap='magma',
    interpolation='nearest'
)
plt.colorbar(label='Значение', fraction=0.02, pad=0.04)
plt.xlabel('Время, кадры')
plt.ylabel('MFCC-коэффициент')
plt.title('MFCC-спектрограмма', pad=12, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Вычисление MFCC готовым методом torchaudio

https://docs.pytorch.org/audio/main/tutorials/audio_feature_extractions_tutorial.html#mfcc

In [ ]:
mfcc_transform = None

mfcc_torch = mfcc_transform(wav)

In [ ]:
plt.figure(figsize=(12, 4))
plt.imshow(
    mfcc_torch.squeeze(),
    aspect='auto',
    origin='lower',
    #cmap='magma',
    interpolation='nearest'
)
plt.colorbar(label='Значение', fraction=0.02, pad=0.04)
plt.xlabel('Время, кадры')
plt.ylabel('MFCC-коэффициент')
plt.title('MFCC-спектрограмма', pad=12, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Cделайте вывод по выполненной работе:

## 2. Работа с фазой аудио
Фаза несёт информацию о точном расположении импульсов во времени и критична для разборчивости речи и естественности звука.
- `2.1` Замена фазы на случайную
- `2.2` Подстановка фазы из другого аудио
- `2.3` Восстановление фазы алгоритмом Griffin-Lim

### 2.1 Удаление оригинальной фазы → замена на случайную

Реализуйте функцию для восстановления raw-audio

In [ ]:
def reconstruct_audio(magnitude, phase):
    pass

In [ ]:
# Магнитуда и фаза исходного сигнала
magnitude = spectrum.abs()
phase = spectrum.angle()

#### Равномерное распределение фазы
Для замены оригинальной фазы на случайную используем **равномерное распределение** на интервале $[-\pi, \pi]$:
$$
\phi_{\text{rand}}[m, k] \sim \mathcal{U}(-\pi, \pi)
$$

In [ ]:
phase_rand = None

In [ ]:
wav_random_phase = reconstruct_audio(magnitude, phase_rand)

print("🎧 Оригинал:")
visualize_audio(wav, SR)
print("🎧 Случайная фаза (магнитуда сохранена):")
visualize_audio(wav_random_phase, SR)

Посчитаем спектр изкаженного аудио

In [ ]:
spectrum_rand_phase = None

In [ ]:
plt.figure(figsize=(10, 3))
plt.imshow(spectrum_rand_phase.abs().squeeze().log(), aspect='auto', origin='lower')
plt.title("Спектрограмма при случайной фазе")
plt.colorbar()
plt.xlabel("Кадр"); plt.ylabel("Частота")
plt.show()

### 3.2 Замена фазы на фазу из другого аудио

Загрузим вторую аудиозапись

In [ ]:
wav2, sr2 = torchaudio.load('female_1.wav')

if sr2 != SR:
    wav2 = torchaudio.transforms.Resample(sr2, SR)(wav2)

In [ ]:
visualize_audio(wav2, sr2)

Вычисляем STFT для второго файла

In [ ]:
spectrum2 = None

magnitude2 = spectrum2.abs()
phase2 = spectrum2.angle()

In [ ]:
# Приводим к одной длине по времени
min_time = min(magnitude.shape[-1], phase2.shape[-1])
mag1_crop = magnitude[..., :min_time]
phase2_crop = phase2[..., :min_time]

Реконструируем аудио с заменненой фазой

In [ ]:
wav_cross_phase = reconstruct_audio(mag1_crop, phase2_crop)

print("🎧 Оригинал 1 (магнитуда) + Фаза аудио 2:")
visualize_audio(wav_cross_phase, SR)

Вычислим STFT искаженного аудио

In [ ]:
spectrum_other_phase = None

In [ ]:
plt.figure(figsize=(10, 3))
plt.imshow(spectrum_other_phase.abs().log().squeeze(), aspect='auto', origin='lower')
plt.title("Фаза из второго аудио, подставленная в первый сигнал")
plt.colorbar(); plt.xlabel("Кадр"); plt.ylabel("Частота")
plt.show()

### 3.3 Простая реализация Griffin-Lim
Алгоритм итеративно уточняет фазу, чередуя прямое и обратное преобразование Фурье.

###  Математическое описание алгоритма Griffin-Lim

Пусть $Y \in \mathbb{R}^{F \times T}$ — заданная целевая амплитудная спектрограмма (magnitude), где $F$ — число частотных бинов, $T$ — число временных кадров. Требуется найти фазовую матрицу $\Phi$, чтобы восстановить сигнал во временной области.

### 1. Инициализация
Фаза задаётся случайным образом на отрезке $[-\pi, \pi]$:
$$ \phi_0[m, k] \sim \mathcal{U}(-\pi, \pi), \quad \forall m \in \{0..T-1\}, \; k \in \{0..F-1\} $$

### 2. Итерационный процесс ($t = 1, \dots, N_{\text{iter}}$)
На каждой итерации выполняются 4 шага:

1. **Формирование комплексного спектра** с сохранением целевой амплитуды:
   $$ \hat{X}_t[m, k] = Y[m, k] \cdot e^{j \phi_{t-1}[m, k]} $$

2. **Обратное преобразование (ISTFT)** → временной сигнал:
   $$ \hat{x}_t[n] = \text{ISTFT}_{w, H, N}\Big( \hat{X}_t[m, k] \Big) $$
   *(где $w[n]$ — окно, $H$ — шаг, $N$ — длина БПФ)*

3. **Прямое преобразование (STFT)** → новый комплексный спектр:
   $$ X'_t[m, k] = \text{STFT}_{w, H, N}\Big( \hat{x}_t[n] \Big) $$

4. **Обновление фазы** (извлекаем аргумент нового спектра):
   $$ \phi_t[m, k] = \arg\Big( X'_t[m, k] \Big) = \angle X'_t[m, k] $$

### 3. Метрика сходимости
Для контроля качества на каждой итерации вычисляется среднеквадратичная ошибка между восстановленной и целевой амплитудами:
$$ \text{MSE}_t = \frac{1}{F \cdot T} \sum_{m=0}^{T-1} \sum_{k=0}^{F-1} \Big( |X'_t[m, k]| - Y[m, k] \Big)^2 $$

### 4. Результат
После $N_{\text{iter}}$ итераций выходным сигналом считается последний восстановленный временной ряд:
$$ \hat{x}_{\text{out}}[n] = \hat{x}_{N_{\text{iter}}}[n] $$

> 💡 **Интуиция:** Алгоритм попеременно проецирует оценку сигнала на два множества:
> 🔹 *Множество временной согласованности* (ISTFT → STFT)
> 🔹 *Множество заданной амплитуды* (замена $|X'|$ на $Y$)
> При достаточном числе итераций процесс сходится к локальному минимуму ошибки амплитуды, восстанавливая правдоподобную фазу.

In [ ]:
def griffin_lim(magnitude, n_iter=100, hop=HOP_LENGTH, win=WIN_LENGTH, n_fft=N_FFT, win_func=WINDOW):
    pass

In [ ]:
# Запускаем на магнитуде первого аудио
wav_griffin = griffin_lim(magnitude, n_iter=30)

print("\n🎧 Восстановление через Griffin-Lim (только магнитуда → фаза):")
visualize_audio(wav_griffin, SR)

Вычисляем STFT для сигнала с восстановленной фазой

In [ ]:
spectrum_griffin_lim = None

In [ ]:
plt.figure(figsize=(10, 3))
plt.imshow(spectrum_griffin_lim.abs().log().squeeze(), aspect='auto', origin='lower')
plt.title("Фаза из второго аудио, подставленная в первый сигнал")
plt.colorbar()
plt.xlabel("Кадр")
plt.ylabel("Частота")
plt.show()

Сравните аудиозаписи по звучанию, и сделайте вывод: